# Add Guardrails & Access Control

This notebook adds two governance layers to the Gateway:
1. **Bedrock Guardrails** — screen tool output for PII and harmful content
2. **Group-based access control** — restrict which tools each team can call

In [ ]:
%pip install "boto3==1.42.87" "httpx==0.27.0" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

## Part A: Bedrock Guardrails

### Step 1: Create a Guardrail

In [ ]:
import boto3, json, base64, httpx

region = boto3.session.Session().region_name or "us-west-2"
bedrock = boto3.client("bedrock", region_name=region)
lam = boto3.client("lambda", region_name=region)

GUARDRAIL_NAME = "workshop-tool-output-guardrail"

# Check for existing guardrail
existing = bedrock.list_guardrails(maxResults=100).get("guardrails", [])
match = [g for g in existing if g["name"] == GUARDRAIL_NAME]

if match:
    GUARDRAIL_ID = match[0]["id"]
    print(f"Guardrail already exists: {GUARDRAIL_ID}")
else:
    resp = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="Screens tool outputs for PII and harmful content",
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
                {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
            ]
        },
        contentPolicyConfig={
            "filtersConfig": [
                {"type": t, "inputStrength": "HIGH", "outputStrength": "HIGH"}
                for t in ["HATE", "VIOLENCE", "SEXUAL", "INSULTS", "MISCONDUCT"]
            ]
        },
        blockedInputMessaging="Input blocked by guardrail.",
        blockedOutputsMessaging="Output blocked: sensitive information detected.",
    )
    GUARDRAIL_ID = resp["guardrailId"]
    print(f"Created guardrail: {GUARDRAIL_ID}")

print(f"Guardrail ID: {GUARDRAIL_ID}")

### Step 2: Attach Guardrail to Response Interceptor

Update the Lambda's environment variable so it screens every tool response.

In [ ]:
FN_NAME = "ac-gateway-response-interceptor"

# Read current env vars
current = lam.get_function_configuration(FunctionName=FN_NAME)
env_vars = current.get("Environment", {}).get("Variables", {})

# Merge guardrail settings
env_vars["BEDROCK_GUARDRAIL_ID"] = GUARDRAIL_ID
env_vars["BEDROCK_GUARDRAIL_VERSION"] = "DRAFT"

# Update
lam.update_function_configuration(
    FunctionName=FN_NAME,
    Environment={"Variables": env_vars},
)
lam.get_waiter("function_updated_v2").wait(FunctionName=FN_NAME)
print(f"Updated {FN_NAME}:")
print(f"  BEDROCK_GUARDRAIL_ID = {GUARDRAIL_ID}")
print(f"  BEDROCK_GUARDRAIL_VERSION = DRAFT")

### Step 3: Test Guardrail Directly

Before testing through the Gateway, verify the guardrail screens PII.

In [ ]:
bedrock_rt = boto3.client("bedrock-runtime", region_name=region)

# Synthetic test data — standard placeholder values only (example.com address,
# 555 phone number, 123-45-6789 SSN). No real PII is used in this workshop.
test_text = "Contact John at john.doe@example.com or call 555-123-4567. SSN: 123-45-6789."

resp = bedrock_rt.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion="DRAFT",
    source="OUTPUT",
    content=[{"text": {"text": test_text}}],
)

print(f"Action: {resp['action']}")
if resp["action"] == "GUARDRAIL_INTERVENED":
    print(f"Blocked/Anonymized output:")
    for output in resp.get("outputs", []):
        print(f"  {output['text']}")
else:
    raise AssertionError(
        f"Expected GUARDRAIL_INTERVENED for text containing email/phone/SSN, "
        f"got action={resp['action']}. The PII filter is not configured correctly."
    )

### Step 4: Test via Gateway

Now call a tool through the Gateway and see the guardrail in action.

In [ ]:
cfn = boto3.client("cloudformation", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

GATEWAY_URL = exports["ac-GatewayUrl"]
TOKEN_URL = exports["workshop-CognitoTokenUrl"]
M2M_SECRET_ARN = exports["workshop-CognitoM2MClientSecretArn"]

sm = boto3.client("secretsmanager", region_name=region)
secret = json.loads(sm.get_secret_value(SecretId=M2M_SECRET_ARN)["SecretString"])

auth_header = base64.b64encode(f"{secret['client_id']}:{secret['client_secret']}".encode()).decode()
token_resp = httpx.post(
    TOKEN_URL,
    headers={"Authorization": f"Basic {auth_header}", "Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials", "scope": "mcp-servers-unrestricted/read mcp-servers-unrestricted/execute"},
    timeout=15,
)
token_resp.raise_for_status()
ACCESS_TOKEN = token_resp.json()["access_token"]
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}", "Content-Type": "application/json"}

# First discover the gateway-prefixed tool name
list_resp = httpx.post(
    GATEWAY_URL, headers=HEADERS,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1}, timeout=30,
)
list_resp.raise_for_status()
tools = list_resp.json().get("result", {}).get("tools", [])
kb_tool = next((t["name"] for t in tools if "knowledge" in t["name"].lower()), "search-knowledge-base")
print(f"Using tool: {kb_tool}")

# Call the knowledge base tool (response will pass through guardrail screening)
resp = httpx.post(
    GATEWAY_URL,
    headers=HEADERS,
    json={
        "jsonrpc": "2.0", "method": "tools/call", "id": 5,
        "params": {"name": kb_tool, "arguments": {"query": "account management", "max_results": 3}}
    },
    timeout=30,
)
resp.raise_for_status()
result = resp.json()
content = result.get("result", {}).get("content", [])
print("Gateway response (with guardrail screening):")
for item in content:
    if item.get("type") == "text":
        print(item["text"])

## Part B: Group-Based Access Control

### Step 5: Set Access Policy

Define which Cognito groups can call which tools.

In [ ]:
REQ_FN = "ac-gateway-request-interceptor"

# Policy keys are Cognito group names. The workshop Cognito User Pool
# (data-stack.yaml) ships with the 'mcp-registry-admin' group.
# '_default' is the fallback applied when a caller's token has no
# matching group claim — e.g. M2M tokens from client_credentials grants,
# which is what the rest of the module uses. Keeping it as ["*"] ensures
# subsequent notebook cells continue to work.
policy = {
    "_default": ["*"],
    "mcp-registry-admin": ["*"],
}

current = lam.get_function_configuration(FunctionName=REQ_FN)
env_vars = current.get("Environment", {}).get("Variables", {})
env_vars["TOOL_ACCESS_POLICY"] = json.dumps(policy)

lam.update_function_configuration(
    FunctionName=REQ_FN,
    Environment={"Variables": env_vars},
)
lam.get_waiter("function_updated_v2").wait(FunctionName=REQ_FN)
print(f"Updated {REQ_FN}:")
print(f"  Policy: {json.dumps(policy, indent=2)}")
print()
print("mcp-registry-admin: Can call ALL tools")
print("_default (no group): Can call ALL tools (M2M callers land here)")

### Step 6: Test Access Control

The M2M client token doesn't include Cognito groups by default (client_credentials flow). To test group-based filtering, you would need a user token from a user assigned to a specific group.

For now, verify the policy is set and understand the mechanism.

In [ ]:
config = lam.get_function_configuration(FunctionName=REQ_FN)
current_policy = config.get("Environment", {}).get("Variables", {}).get("TOOL_ACCESS_POLICY", "")
if current_policy:
    parsed = json.loads(current_policy)
    print("Current access policy:")
    for group, patterns in parsed.items():
        print(f"  {group}: {patterns}")
else:
    print("No access policy set (all tools accessible to all callers)")

## Part C: Policy Engine (Cedar Policies)

The interceptor-based ACL above uses custom Lambda logic. AgentCore also provides a **Policy Engine** — a managed service that evaluates [Cedar](https://www.cedarpolicy.com/) policies. Cedar is a declarative, auditable language purpose-built for authorization.

This gives you two complementary access control layers:
- **Layer 1 (Interceptor ACL):** Fast, custom logic, fail-open — already configured above
- **Layer 2 (Cedar Policy Engine):** Declarative, AWS-managed, auditable — configured here

### Step 7: Create a Policy Engine

In [ ]:
cp_client = boto3.client("bedrock-agentcore-control", region_name=region)

ENGINE_NAME = "workshop_policy_engine"

# Check for existing engine
existing = cp_client.list_policy_engines().get("policyEngines", [])
engine_match = [e for e in existing if e.get("name") == ENGINE_NAME]

if engine_match:
    ENGINE_ID = engine_match[0]["policyEngineId"]
    print(f"Policy Engine already exists: {ENGINE_ID}")
else:
    resp = cp_client.create_policy_engine(name=ENGINE_NAME)
    ENGINE_ID = resp["policyEngineId"]
    print(f"Created Policy Engine: {ENGINE_ID}")

print(f"Engine ID: {ENGINE_ID}")

### Step 8: Create Cedar Policies

The Cedar schema maps to your Gateway automatically:
- **Actions** = Gateway targets (`AgentCore::Action::"tg-workshop-flights-mcp"`)
- **Resource** = Gateway ARN (`AgentCore::Gateway::"arn:..."`)
- **Principals** = `AgentCore::OAuthUser` (JWT) or `AgentCore::IamEntity` (IAM)

Define two policies:
1. **Developer tool access** — only flights + knowledge-base targets
2. **Admin full access** — all 3 targets explicitly (wildcard actions are rejected as "Overly Permissive")

In [ ]:
# Get the Gateway ARN for Cedar resource constraints
GATEWAY_ID = exports.get("ac-GatewayId", "")
ACCOUNT_ID = boto3.client("sts", region_name=region).get_caller_identity()["Account"]
GATEWAY_ARN = f"arn:aws:bedrock-agentcore:{region}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}"

# Discover gateway targets (these become Cedar actions)
targets = cp_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])
print("Gateway targets (Cedar actions):")
for t in targets:
    print(f'  AgentCore::Action::"{t["name"]}"')

# Build Cedar policies from actual discovered target names (not hardcoded — target
# names have differed between environments, e.g. tg-workshop-kb-search vs tg-search-knowledge-base).
target_names = [t["name"] for t in targets]
if not target_names:
    raise RuntimeError(
        "No gateway targets discovered — cannot build Cedar policies. "
        "Re-run notebook 02-deploy to ensure the gateway and its targets exist."
    )

def _find_target(substring):
    match = [n for n in target_names if substring in n.lower()]
    return match[0] if match else None

flights_target = _find_target("flights")
hotels_target = _find_target("hotels")
kb_target = _find_target("knowledge") or _find_target("kb-search") or _find_target("kb")
missing = [label for label, v in [("flights", flights_target), ("hotels", hotels_target), ("knowledge-base", kb_target)] if not v]
if missing:
    raise RuntimeError(
        f"Expected gateway targets not found for: {missing}. Discovered: {target_names}. "
        f"The Cedar policies in this cell assume flights/hotels/knowledge-base targets exist."
    )

# Cedar policy: Developers can only call flights + knowledge-base targets
DEVELOPER_POLICY = f"""
permit(
    principal,
    action in [
        AgentCore::Action::"{flights_target}",
        AgentCore::Action::"{kb_target}"
    ],
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
);
""".strip()

# Cedar policy: Admins can call all 3 targets (wildcards rejected as "Overly Permissive")
ADMIN_POLICY = f"""
permit(
    principal,
    action in [
        AgentCore::Action::"{flights_target}",
        AgentCore::Action::"{hotels_target}",
        AgentCore::Action::"{kb_target}"
    ],
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
);
""".strip()

policies = [
    ("developer_tool_access", DEVELOPER_POLICY, "Developers: flights + search-kb only"),
    ("admin_full_access", ADMIN_POLICY, "Admins: all 3 targets"),
]

policy_ids = {}
for name, statement, desc in policies:
    try:
        resp = cp_client.create_policy(
            policyEngineId=ENGINE_ID,
            name=name,
            definition={"cedar": {"statement": statement}},
        )
        policy_id = resp["policyId"]
        policy_ids[name] = policy_id
        print(f"Created policy: {name} ({policy_id})")
    except cp_client.exceptions.ConflictException:
        print(f"Policy {name} already exists -- skipping")

print(f"\nCedar policies define WHO can call WHAT -- declaratively and auditably.")
print(f"Developer policy:\n{DEVELOPER_POLICY}")

### Step 9: Verify Policies

List all policies in the engine to confirm they were created.

In [ ]:
all_policies = cp_client.list_policies(policyEngineId=ENGINE_ID).get("policies", [])

print(f"Policy Engine: {ENGINE_ID}")
print(f"Policies: {len(all_policies)}")
print()
for p in all_policies:
    print(f"  Name:   {p.get('name', '?')}")
    print(f"  ID:     {p.get('policyId', '?')}")
    print(f"  Status: {p.get('status', '?')}")
    print()

### Step 10: Attach the Policy Engine to the Gateway

Creating a Policy Engine is not enough — it only becomes enforced when attached to a Gateway via `update_gateway` with `policyEngineConfiguration`. We read the Gateway's current config, merge in the Policy Engine ARN in `ENFORCE` mode, and push the update.

In [ ]:
GATEWAY_ID = exports["ac-GatewayId"]
POLICY_ENGINE_ARN = f"arn:aws:bedrock-agentcore:{region}:{ACCOUNT_ID}:policy-engine/{ENGINE_ID}"

gw = cp_client.get_gateway(gatewayIdentifier=GATEWAY_ID)

cp_client.update_gateway(
    gatewayIdentifier=GATEWAY_ID,
    name=gw["name"],
    roleArn=gw["roleArn"],
    protocolType=gw["protocolType"],
    authorizerType=gw["authorizerType"],
    authorizerConfiguration=gw["authorizerConfiguration"],
    policyEngineConfiguration={"mode": "ENFORCE", "arn": POLICY_ENGINE_ARN},
)

# Poll until the Gateway is READY again after the update
import time as _time
gw_status = "UNKNOWN"
for attempt in range(12):  # up to 60s
    g = cp_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
    gw_status = g.get("status", "UNKNOWN")
    if gw_status == "READY":
        break
    print(f"  [{(attempt+1)*5}s] Gateway status: {gw_status}")
    _time.sleep(5)
else:
    raise RuntimeError(
        f"Gateway {GATEWAY_ID} did not return to READY after Policy Engine attach "
        f"(last status: {gw_status}). Subsequent tool calls will fail."
    )

print(f"Attached Policy Engine to Gateway {GATEWAY_ID}")
print(f"  Mode: ENFORCE")
print(f"  ARN:  {POLICY_ENGINE_ARN}")
print(f"  Gateway status: {gw_status}")
print()
print("Cedar policies are now evaluated on every tool call through this Gateway.")

## Summary

| Governance Layer | Mechanism | Effect |
|---|---|---|
| **PII Screening** | Bedrock Guardrail on response interceptor | Anonymize/block PII in tool outputs |
| **Access Control L1** | TOOL_ACCESS_POLICY on request interceptor | Filter tools by Cognito group (custom Lambda logic) |
| **Access Control L2** | Cedar policies in Policy Engine | Declarative, auditable tool access rules (AWS-managed) |

The platform now has three independent governance layers:
1. **Guardrails** screen content on the way out
2. **Interceptor ACL** filters tools on the way in (fast, custom)
3. **Cedar Policy Engine** provides declarative authorization (auditable, managed)

Individual tool Lambdas don't implement any security logic — all governance is enforced by the platform.

**Next:** Notebook 08 cleans up all resources.